To generate the entanglement (Gauss linking number) traces, run:

```
python3 compute_entanglement.py -j8
```

This will create one `.txt` file per replicate under an `entanglement/` subfolder
next to the replicate folders, analogous to `partitioning/` for `compute_partitioning.py`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def load_entanglement_file(filepath):
    """Load minute and entanglement (linking number) arrays from one replicate .txt file."""
    minutes, ent = [], []
    with open(filepath) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith("#") or not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                minutes.append(int(parts[0]))
                ent.append(float(parts[1]))
    return (np.array(minutes), np.array(ent)) if minutes else (np.array([]), np.array([]))

def plot_entanglement_trajectories(dates=None, replicates=None, entanglement_dir=None, base_dir=None, ax=None,
                                   title="Daughter chromosome entanglement (Gauss linking number) vs time",
                                   figsize=(8, 5), labels=None, ylim=None):
    """
    Load entanglement .txt files and plot one trace per replicate.

    labels: optional dict mapping replicate name -> legend label
            (e.g. {"Feb08_1": "1× speed"}); missing keys use replicate name.
    ylim: optional (ymin, ymax) for y-axis limits; default uses data range.
    """
    base_dir = Path(base_dir) if base_dir is not None else Path(".")
    ent_dir = Path(entanglement_dir) if entanglement_dir is not None else base_dir / "entanglement"
    if replicates is None:
        if not dates:
            raise ValueError("Provide either dates or replicates")
        replicates = sorted(
            f.stem for f in ent_dir.glob("*.txt")
            if any(f.stem.startswith(d) for d in dates)
        )
    if not replicates:
        raise FileNotFoundError(f"No entanglement files found for dates {dates} in {ent_dir}")
    series = {}
    for rep in replicates:
        path = ent_dir / f"{rep}.txt"
        if path.exists():
            minutes, ent = load_entanglement_file(path)
            if len(minutes) > 0:
                series[rep] = (minutes, ent)
    if not series:
        raise FileNotFoundError(f"No data loaded for {replicates} in {ent_dir}")
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig, created_fig = None, False
    colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(series))))
    for k, rep in enumerate(sorted(series.keys())):
        minutes, ent = series[rep]
        order = np.argsort(minutes)
        trace_label = labels.get(rep, rep) if labels else rep
        ax.plot(minutes[order], ent[order], label=trace_label, color=colors[k % len(colors)])
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Gauss linking number")
    if ylim is not None:
        ymin = ylim[0] if ylim[0] is not None else None
        ymax = ylim[1] if len(ylim) > 1 and ylim[1] is not None else None
        ax.set_ylim(ymin, ymax)
    ax.legend()
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if created_fig:
        plt.tight_layout()
    return (fig, ax) if created_fig else (None, ax)

# Example usage (adjust base_dir and dates/replicates as needed):
base_dir = Path("/raid/amaytin/protein_science/runs/")
# fig, ax = plot_entanglement_trajectories(dates=["Feb16"], base_dir=base_dir)
# plt.show()

In [ ]:
# --- 3D sweep runs table (matches analysis/runs.ipynb and submit_jobs_3d_sweep.sh) ---
import pandas as pd
import itertools

# Parameter grid: must match runs.ipynb and submit_jobs_3d_sweep.sh
N_VALUES = [20, 25, 30, 35, 50]
V_VALUES = [200, 350, 500]
TAU_VALUES = [45, 65, 85, 105]
# Run name prefix for 3D sweep (e.g. Feb16 for submit_jobs_3d_sweep)
RUN_DATE = "Feb16"

all_runs = list(itertools.product(N_VALUES, V_VALUES, TAU_VALUES))
runs_list = []
for run_id, (N, v, tau) in enumerate(all_runs, 1):
    S = N * v * tau
    run_name = f"{RUN_DATE}_p{run_id}"
    runs_list.append({"run": run_id, "run_name": run_name, "N": N, "v": v, "tau": tau, "S": S})

runs_df = pd.DataFrame(runs_list)

def filter_runs(runs_df, run_ids=None, N=None, v=None, tau=None):
    """Return subset of runs by run ID(s), N, v, and/or tau.

    E.g. filter_runs(runs_df, N=20) for all runs with N=20.
    """
    out = runs_df.copy()
    if run_ids is not None:
        run_ids = [run_ids] if np.isscalar(run_ids) else list(run_ids)
        out = out[out["run"].isin(run_ids)]
    if N is not None:
        out = out[out["N"] == N]
    if v is not None:
        out = out[out["v"] == v]
    if tau is not None:
        out = out[out["tau"] == tau]
    return out.reset_index(drop=True)

# Example: subset with N=20
# filter_runs(runs_df, N=20)
# filter_runs(runs_df, run_ids=[1, 2, 3])
# filter_runs(runs_df, v=350)

runs_df

In [ ]:
def plot_entanglement_subset(runs_df, run_ids=None, N=None, v=None, tau=None, base_dir=None,
                              entanglement_dir=None, title=None, labels=None, figsize=(8, 5),
                              ylim=None):
    """Plot entanglement trajectories for a subset of the 48 runs.

    Filter by run_ids (int or list), N, v, and/or tau.
    E.g. plot_entanglement_subset(runs_df, N=20)  # all runs with N=20
         plot_entanglement_subset(runs_df, run_ids=[1, 5, 9], title="...")
    """
    subset = filter_runs(runs_df, run_ids=run_ids, N=N, v=v, tau=tau)
    if subset.empty:
        raise ValueError("No runs match the filter")
    replicates = subset["run_name"].tolist()
    if labels is None:
        labels = {row["run_name"]: f"N={row['N']}, v={row['v']}, tau={row['tau']}" for _, row in subset.iterrows()}
    if title is None:
        parts = []
        if N is not None:
            parts.append(f"N={N}")
        if v is not None:
            parts.append(f"v={v}")
        if tau is not None:
            parts.append(f"tau={tau}")
        if run_ids is not None:
            parts.append(f"run_ids={run_ids}")
        title = "Entanglement vs time  " + (", ".join(parts) if parts else f"{len(replicates)} runs")
    fig, ax = plot_entanglement_trajectories(
        replicates=replicates,
        entanglement_dir=entanglement_dir,
        base_dir=base_dir,
        title=title,
        figsize=figsize,
        labels=labels,
        ylim=ylim,
    )
    return fig, ax


def get_entanglement_at_minute(rep_name, minute, ent_dir=None, base_dir=None):
    """Load entanglement file for one run and return value at given minute (interpolate if needed)."""
    ent_dir = Path(ent_dir) if ent_dir is not None else Path(base_dir) / "entanglement"
    path = ent_dir / f"{rep_name}.txt"
    if not path.exists():
        return np.nan
    minutes, ent = load_entanglement_file(path)
    if len(minutes) == 0:
        return np.nan
    if minute in minutes:
        return ent[minutes == minute][0]
    # Linear interpolation
    order = np.argsort(minutes)
    minutes = minutes[order]
    ent = ent[order]
    if minute <= minutes[0]:
        return ent[0]
    if minute >= minutes[-1]:
        return ent[-1]
    return np.interp(minute, minutes, ent)


# Marker options for differentiating N or tau
_MARKERS = ["o", "s", "^", "D", "v", "<", ">", "p", "*", "h", "H", "X"]


def _default_encoding_for_x_axis(x_axis):
    """Sensible defaults for color_by, marker_by, size_by when plotting vs x_axis."""
    params = ["N", "v", "tau"]
    others = [p for p in params if p != x_axis]
    if x_axis == "S":
        return {"color_by": "v", "marker_by": "N", "size_by": "tau"}
    if len(others) == 2:
        return {"color_by": others[0], "marker_by": others[1], "size_by": None}
    return {"color_by": others[0], "marker_by": others[1], "size_by": others[2]}


def plot_entanglement_vs(runs_df, x_axis="S", minute=60, run_ids=None, N=None, v=None, tau=None,
                          base_dir=None, entanglement_dir=None, ax=None, title=None, figsize=(8, 5),
                          color_by=None, marker_by=None, size_by=None, show_legend=True,
                          size_range=(35, 120)):
    """Plot extent of entanglement at a given minute vs N, v, tau, or S.

    x_axis: "N", "v", "tau", or "S"  quantity on the x-axis.
    Use color_by, marker_by, size_by (each "N", "v", or "tau"; or None) to distinguish parameters.
    Defaults: if not set, chosen from the other dimensions (e.g. when x_axis="v": color_by="tau", marker_by="N").
    size_range = (min_s, max_s) for size_by.
    """
    if x_axis not in ("N", "v", "tau", "S"):
        raise ValueError("x_axis must be one of 'N', 'v', 'tau', 'S'")
    defaults = _default_encoding_for_x_axis(x_axis)
    if color_by is None:
        color_by = defaults["color_by"]
    if marker_by is None:
        marker_by = defaults["marker_by"]
    if size_by is None:
        size_by = defaults["size_by"]

    subset = filter_runs(runs_df, run_ids=run_ids, N=N, v=v, tau=tau)
    if subset.empty:
        raise ValueError("No runs match the filter")
    ent_dir = Path(entanglement_dir) if entanglement_dir is not None else Path(base_dir) / "entanglement"
    ent_at_t = []
    for _, row in subset.iterrows():
        e = get_entanglement_at_minute(row["run_name"], minute, ent_dir=ent_dir, base_dir=base_dir)
        ent_at_t.append(e)
    subset = subset.copy()
    subset["entanglement"] = ent_at_t
    subset = subset.dropna(subset=["entanglement"])
    if subset.empty:
        raise FileNotFoundError(
            f"No entanglement data at minute {minute} for the selected runs in {ent_dir}"
        )

    x_col = x_axis
    x_label = "S (N × v × tau)" if x_axis == "S" else x_axis

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig, created_fig = None, False

    color_vals = sorted(subset[color_by].unique()) if color_by and color_by in subset.columns else [None]
    marker_vals = sorted(subset[marker_by].unique()) if (marker_by and marker_by in subset.columns) else [None]
    size_vals = sorted(subset[size_by].unique()) if (size_by and size_by in subset.columns) else [None]
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(color_vals), 1)))
    markers = [_MARKERS[i % len(_MARKERS)] for i in range(len(marker_vals))]
    s_min, s_max = size_range
    sizes = np.linspace(s_min, s_max, len(size_vals)) if size_vals and size_vals[0] is not None else [60]

    for ci, cval in enumerate(color_vals):
        for mi, mval in enumerate(marker_vals):
            for si, sval in enumerate(size_vals):
                mask = (subset[color_by] == cval) if (color_by and cval is not None) else np.ones(len(subset), dtype=bool)
                if marker_by and mval is not None:
                    mask = mask & (subset[marker_by] == mval)
                if size_by and sval is not None:
                    mask = mask & (subset[size_by] == sval)
                if not mask.any():
                    continue
                lbl = []
                if color_by and cval is not None:
                    lbl.append(f"{color_by}={cval}")
                if marker_by and mval is not None:
                    lbl.append(f"{marker_by}={mval}")
                if size_by and sval is not None:
                    lbl.append(f"{size_by}={sval}")
                label = ", ".join(lbl) if lbl else None
                pt_size = int(sizes[si]) if (size_by and sval is not None) else 60
                ax.scatter(
                    subset.loc[mask, x_col], subset.loc[mask, "entanglement"],
                    c=[colors[ci]] if (color_by and cval is not None) else None,
                    marker=markers[mi] if (marker_by and mval is not None) else "o",
                    s=pt_size, label=label, alpha=0.85, edgecolors="k", linewidths=0.5,
                )

    if show_legend:
        ax.legend(loc="best")

    ax.set_xlabel(x_label)
    ax.set_ylabel(f"Entanglement at {minute} min (Gauss Lk)")
    if title is None:
        title = f"Entanglement at {minute} min vs {x_label}"
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if created_fig:
        plt.tight_layout()
    return (fig, ax) if created_fig else (None, ax)


def plot_entanglement_vs_S(
    runs_df,
    minute=60,
    run_ids=None,
    N=None,
    v=None,
    tau=None,
    base_dir=None,
    entanglement_dir=None,
    ax=None,
    title="Extent of entanglement at 60 s vs S",
    figsize=(8, 5),
    color_by="v",
    marker_by="N",
    size_by="tau",
    show_legend=True,
    size_range=(35, 120),
):
    """Convenience wrapper to plot entanglement at a given minute vs S = N*v*tau."""
    return plot_entanglement_vs(
        runs_df,
        x_axis="S",
        minute=minute,
        run_ids=run_ids,
        N=N,
        v=v,
        tau=tau,
        base_dir=base_dir,
        entanglement_dir=entanglement_dir,
        ax=ax,
        title=title,
        figsize=figsize,
        color_by=color_by,
        marker_by=marker_by,
        size_by=size_by,
        show_legend=show_legend,
        size_range=size_range,
    )


In [ ]:
fig, ax = plot_entanglement_subset(runs_df, N=20, base_dir=base_dir)
plt.show()